# Phase 1: การเตรียมข้อมูล (Data Preprocessing)
ส่วนนี้แสดงกระบวนการนำเข้าชุดข้อมูล (Data Import) การจัดเตรียมโครงสร้างข้อมูล และการสร้างตัวแปรเป้าหมาย (Target Variable: `risk_level`) ด้วยระบบประเมินคะแนนความเสี่ยง (Risk Scoring)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ตั้งค่าฟอนต์ภาษาไทยสำหรับกราฟ
plt.rcParams['font.family'] = 'Tahoma'

# 1. การนำเข้าข้อมูล
df = pd.read_csv('income_expense_behavior_vs_cost_of_living_crisis.csv')
print(f'ขนาดของข้อมูลเริ่มต้น: {df.shape}')
df.head(2)

In [ ]:
# 2. การเปลี่ยนชื่อตัวแปรให้เป็นภาษาอังกฤษ
column_mapping = {
    "1. เพศของท่าน": "sex",
    "2. ช่วงอายุเเละช่วงวัย": "age_group",
    "3. สถานะการทำงานของท่าน": "occupation",
    "4. รายได้สุทธิเฉลี่ยต่อเดือน (หลังหักภาษี)": "income",
    "5. จำนวนสมาชิกในครอบครัวที่อยู่ในความดูเเลรับผิดชอบด้านค่าใช้จ่าย (รวมตัวท่าน) ": "family_size",
    "6. ภาระหนี้สินคงที่ต่อเดือน": "debt",
    "7. ค่าใช้จ่ายด้านน้ำมันเเละการเดินทางเฉลี่ยต่อเดือน (บาท)": "travel_cost",
    "8. จำนวนผลิตภัณฑ์บัตรเครดิตหรือวงเงินสินเชื่อหมุนเวียนที่ถือครอง": "credit_cards",
    "9. ลักษณะการชำระหนี้บัตรเครดิตในรอบ 6 เดือนที่ผ่านมา": "payment_behavior",
    " 10. ความถี่ในการเลือกซื้อสินค้าตามความพึงพอใจ โดยไม่ได้วางเเผน": "impulse_buying",
    "11. การถือครองทรัพย์สินหรือการลงทุน ": "assets",
    "12. สภาพคล่องคงเหลือ (กรณีรายได้หยุด ชะงัก ท่านมีเงินสำรองอยู่ได้กี่เดือน?)": "liquidity",
    "13. ในรอบ 1 ปีที่ผ่านมา ท่านเคยประสบสภาวะ \"หมุนเงินไม่ทัน\" จนต้องหยิบยืมเงินหรือไม่": "shortage",
    "14. หากเกิดเหตุฉุกเฉินที่ต้องใช้เงินก้อนทันที ท่านมีเเหล่งเงินทุนรองรับหรือไม่": "emergency_fund",
    "15. ท่านมีความกังวลต่อสถานะทางการเงินของตนเองในปัจจุบันระดับใด": "concern_level",
    "16. หากราคาน้ำมันเเละค่าครองชีพสูงขึ้น ร้อยละ 20 ท่านจะให้ความสำคัญกับการดำเนินการสิ่งใดมากที่สุด": "coping_strategy",
    "17. ท่านมีความรู้ความเข้าใจด้านการวางเเผนการเงินเเละภาษี": "financial_literacy",
    "18. ท่านมีสวัสดิการหรือประกันสุขภาพส่วนบุคคลที่เพียงพอต่อความเสี่ยง ในรูปแบบใดบ้าง ": "welfare"
}
df = df.rename(columns=column_mapping)
print("เปลี่ยนชื่อคอลัมน์เรียบร้อยแล้ว")

In [ ]:
# การจัดกลุ่มข้อมูลอายุ (Generation) และรายได้
age_map = {
    'Gen Z (เกิดปี พ.ศ. 2541 - 2555 / อายุ 14 - 28 ปี)': 'Gen Z',
    'Gen Y / Millennials (เกิดปี พ.ศ. 2523 - 2540 / อายุ 29 - 46 ปี)': 'Gen Y',
    'Gen X (เกิดปี พ.ศ. 2508 - 2522 / อายุ 47 - 61 ปี)': 'Gen X'
}
df['age_group'] = df['age_group'].map(age_map)

income_map = {
    'ต่ำกว่า 10,000 บาท': 5000,
    '10,001 - 25,000 บาท': 17500,
    '25,001 - 50,000 บาท': 37500,
    'มากกว่า 50,000 บาท': 75000
}
df['income_num'] = df['income'].map(income_map)

# การจัดการค่าสูญหาย (Missing Values) ด้วยค่ามัธยฐาน (Median)
df['travel_cost'] = pd.to_numeric(df['travel_cost'], errors='coerce')
df['travel_cost'].fillna(df['travel_cost'].median(), inplace=True)

In [ ]:
# 3. สร้างตัวแปรเป้าหมาย (Target Variable)
# เปลี่ยน Target เป็น "ประวัติการหมุนเงินไม่ทัน" (แบ่งเป็น 2 กลุ่ม)
# เพื่อให้โมเดลทำนายได้แม่นยำและนำไปใช้งานได้จริง
# "เสี่ยง" = เคยหมุนเงินไม่ทัน (1-2 ครั้ง หรือ มากกว่า 2 ครั้ง)
# "ปลอดภัย" = ไม่เคยหมุนเงินไม่ทันเลย
def calculate_risk(shortage_text):
    if shortage_text == 'ไม่เคย':
        return 'Safe'
    else:
        return 'At Risk'

df['risk_level'] = df['shortage'].apply(calculate_risk)

print('สรุปจำนวนตัวอย่างในแต่ละกลุ่มความเสี่ยง:')
print(df['risk_level'].value_counts())


---
# Phase 2: การวิเคราะห์ Insight และการแสดงผล (EDA & Visualization)
ส่วนนี้เป็นการสร้างการสร้างแผนภาพข้อมูล (Data Visualization) เพื่อค้นหาความสัมพันธ์ของตัวแปรต่างๆ

In [ ]:
# วิเคราะห์ Insight จากข้อมูล
# Insight 1: ใครคือหนี้ก้อนใหญ่? (Debt by Generation)
# วิเคราะห์ Insight จากข้อมูล

# การจัดกลุ่มข้อมูลเพื่อหาความถี่
debt_gen = df.groupby(['age_group', 'debt']).size().unstack()

debt_gen.plot(kind='bar', stacked=True, figsize=(10,6), color=['skyblue', 'salmon', 'lightgreen', 'gold', 'orchid'])
plt.title('Insight 1: Debt by Generation')
plt.ylabel('จำนวนคน (Count)')
plt.xlabel('ช่วงวัย (Generation)')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# วิเคราะห์ Insight จากข้อมูล
# Insight 2: เงินสำรองฉุกเฉิน คือ เกราะกำบังความเสี่ยงตัวจริง?
# สไตล์: Pie Chart 4 อัน (2x2 grid) แยกตามระดับเงินสำรอง
# วิเคราะห์ Insight จากข้อมูล

# ลำดับเงินสำรองจากน้อยไปมาก
liq_order = ['น้อยกว่า 1 เดือน', '1 - 3 เดือน', '3 - 6 เดือน', 'มากกว่า 6 เดือน']
colors_pie = ['indianred', 'mediumseagreen']

fig, axes = plt.subplots(2, 2, figsize=(8, 6))
fig.suptitle('Insight 2: เงินสำรองฉุกเฉิน คือ เกราะกำบังความเสี่ยงตัวจริง?', fontsize=13)

for i, liq in enumerate(liq_order):
    ax = axes[i // 2][i % 2]
    sub = df[df['liquidity'] == liq]
    counts = sub.groupby('risk_level').size().reindex(['At Risk', 'Safe'], fill_value=0)
    ax.pie(counts, labels=['At Risk', 'Safe'], colors=colors_pie, autopct='%1.0f%%', startangle=90)
    ax.set_title(f'สำรอง: {liq}', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# วิเคราะห์ Insight จากข้อมูล
# Insight 3: เงินเดือนเยอะแปลว่ารอดจริงหรือ? (Income vs. Risk Level)
# สไตล์: กราฟแท่งแนวนอน (Horizontal Bar Chart)
# วิเคราะห์ Insight จากข้อมูล

# การจัดกลุ่มช่วงรายได้
income_labels = {
    5000: '<10k',
    17500: '10k-25k',
    37500: '25k-50k',
    75000: '>50k'
}
df['income_label'] = df['income_num'].map(income_labels)
income_order = ['<10k', '10k-25k', '25k-50k', '>50k']

# 1. นับจำนวนคนแยกตาม รายได้ และ ความเสี่ยง
income_risk_count = df.groupby(['income_label', 'risk_level']).size().unstack()

# 2. แปลงเป็นเปอร์เซ็นต์ (หารด้วยผลรวมของแต่ละแถว)
income_risk_percent = income_risk_count.div(income_risk_count.sum(axis=1), axis=0)

# 3. เรียงลำดับช่วงรายได้ให้ถูกต้อง
income_risk_percent = income_risk_percent.reindex(income_order)

colors = {'Safe': 'mediumseagreen', 'At Risk': 'indianred'}
# ปรับเป็นกราฟแนวนอน (barh) และไม่ stack (stacked=False) 
# เพิ่มขอบสีดำ (edgecolor) และเส้นตาราง (grid) แบบพื้นฐาน
income_risk_percent.plot(kind='barh', stacked=False, color=[colors[c] for c in income_risk_percent.columns], 
                         figsize=(10,6), edgecolor='black')
plt.title('Insight 3: เปรียบเทียบสัดส่วนความเสี่ยงในแต่ละช่วงรายได้')
plt.xlabel('สัดส่วน (Percentage)')
plt.ylabel('ช่วงรายได้')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.legend(title='ระดับความเสี่ยง', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
# วิเคราะห์ Insight จากข้อมูล
# Insight 4: แผนรับมือเมื่อวิกฤตมาเยือน (Coping Strategy by Gen)
# สไตล์: กราฟแท่งจัดกลุ่ม (Grouped Bar Chart)
# วิเคราะห์ Insight จากข้อมูล

coping_gen = df.groupby(['coping_strategy', 'age_group']).size().unstack()

coping_gen.plot(kind='bar', figsize=(12,6), colormap='Set2', edgecolor='black')
plt.title('Insight 4: Coping Strategy by Generation')
plt.xlabel('แนวทางการรับมือ')
plt.ylabel('จำนวนคน (Count)')
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
# วิเคราะห์ Insight จากข้อมูล
# Insight 5: ความรู้คือภูมิคุ้มกันชั้นดี? (Financial Literacy vs. Risk Level)
# สไตล์: Boxplot ดูการกระจายตัวของความรู้ทางการเงินในแต่ละกลุ่มความเสี่ยง
# วิเคราะห์ Insight จากข้อมูล

# การจัดเรียงลำดับกลุ่มความเสี่ยงสำหรับการแสดงผล
df['risk_level'] = pd.Categorical(df['risk_level'], categories=['At Risk', 'Safe'], ordered=True)

# ใช้คำสั่งพื้นฐานของ pandas ในการทำ Boxplot
# ปรับแต่งสีของกล่องและเส้นให้ดูดีขึ้นด้วยพารามิเตอร์พื้นฐาน (color props)
# Boxes=น้ำเงิน, Whiskers=ส้ม, Medians=แดง
props = dict(boxes="blue", whiskers="orange", medians="red", caps="gray")
df.boxplot(column='financial_literacy', by='risk_level', grid=False, figsize=(8,5), color=props)

plt.title('Insight 5: การกระจายตัวของความรู้ทางการเงิน เทียบกับความเสี่ยง')
plt.suptitle('') # ลบ title อัตโนมัติของ pandas ทิ้งให้กราฟดูสะอาด
plt.xlabel('ระดับความเสี่ยง')
plt.ylabel('คะแนนความรู้ทางการเงิน (1-5)')
plt.grid(axis='y', linestyle=':', alpha=0.5) # เพิ่มเส้นตารางบางๆ แนวนอน
plt.show()

---
# Phase 3: การพัฒนาและทดสอบโมเดล (Machine Learning)
ส่วนนี้แสดงกระบวนการพัฒนาและทดสอบแบบจำลอง (Model Training and Evaluation) เพื่อจำแนกกลุ่มความเสี่ยงทางการเงิน

**ขั้นตอนที่ 1: การเตรียมข้อมูลให้คอมพิวเตอร์เข้าใจ (Data Encoding)**
คอมพิวเตอร์ไม่รู้จักคำว่า "ไม่มีบัตรเครดิต" เราต้องแปลงคำตอบเหล่านี้เป็น **"ตัวเลข"** ให้หมดก่อน

In [ ]:
# การแปลงข้อมูลเชิงคุณภาพให้เป็นข้อมูลเชิงปริมาณ
liq_score_map = {
      'น้อยกว่า 1 เดือน': 3,
      '1 - 3 เดือน': 2,
      '3 - 6 เดือน': 1,
      'มากกว่า 6 เดือน': 0
}
debt_score_map = {
      'มากกว่าร้อยละ 50 ของรายได้': 3,
      'ร้อยละ 31 - 50 ของรายได้': 2,
      'ต่ำกว่าร้อยละ 30 ของรายได้': 1,
      'ไม่มีภาระหนี้สิน': 0
}
payment_map = {
    'ไม่มีบัตรเครดิต': 0,      'ชำระเต็มจำนวน': 0,
    'ชำระขั้นต่ำบางส่วน': 2,   'ชำระขั้นต่ำทุกงวด': 3,
    'ค้างชำระ': 4
}
credit_map = {
    'ไม่มีบัตรเครดิตหรือวงเงินสินเชื่อ': 0,
    '1 - 2 ใบ': 1, '3 - 4 ใบ': 2, '5 ใบขึ้นไป': 3
}
assets_map = {
    'ไม่มีการถือครอง': 3,
    'มีน้อยกว่า 100 % ของรายได้': 2,
    'มี 100% - 500% ของรายได้': 1,
    'มี 501% - 1,200% ของรายได้': 0,
    'มีมากกว่า 1,200% ของรายได้': 0
}
emergency_map = {
    'มีเงินเก็บส่วนตัวเพียงพอ': 0,
    'ขอความช่วยเหลือทางการเงินจากครอบครัว/ญาติ': 1,
    'ใช้บัตรเครดิต / กดเงินสด / กู้ยืมสินเชื่อในระบบ': 2,
    'หยิบยืมจากเพื่อนหรือคนรู้จัก': 3,
    'กู้ยืมเงินจากแหล่งนอกระบบ': 4
}
family_map = {
    '1 ท่าน (ดูเเลตนเองเพียงลำพัง)': 1,
    '2 ท่าน': 2, '3 ท่าน': 3, '4 ท่าน': 4, '5 ท่านขึ้นไป': 5
}

df['liquidity_score'] = df['liquidity'].map(liq_score_map)
df['debt_score']      = df['debt'].map(debt_score_map)
df['payment_score']   = df['payment_behavior'].map(payment_map)
df['credit_score']    = df['credit_cards'].map(credit_map)
df['assets_score']    = df['assets'].map(assets_map)
df['emergency_score'] = df['emergency_fund'].map(emergency_map)
df['family_num']      = df['family_size'].map(family_map)

# การกำหนดตัวแปรอิสระ (Features / X)
features = [
    'income_num', 'debt_score', 'liquidity_score',
    'payment_score', 'credit_score', 'assets_score', 'emergency_score',
    'impulse_buying', 'concern_level', 'financial_literacy',
    'travel_cost', 'family_num'
]

# การแปลงตัวแปรเป้าหมาย (Target Variable / y) เป็นค่าตัวเลข
risk_encode = {'Safe': 0, 'At Risk': 1}
df['risk_code'] = df['risk_level'].map(risk_encode)

X = df[features]
y = df['risk_code']

print("การจัดเตรียมชุดข้อมูล X และ y เสร็จสมบูรณ์")

**ขั้นตอนที่ 2: แบ่งข้อมูลสำหรับการสอน (Train) และการสอบ (Test)**
เก็บข้อมูล 20% ไว้เป็น "ข้อสอบลับ" เพื่อวัดว่าคอมพิวเตอร์เก่งจริงไหม

In [ ]:
from sklearn.model_selection import train_test_split

# การแบ่งชุดข้อมูล (Training Set 80% และ Testing Set 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ปรับข้อมูลให้เป็นมาตรฐาน (Standardization) เพื่อให้โมเดลคำนวณได้ง่ายขึ้น
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# ใช้ pd.DataFrame คลุมผลลัพธ์เพื่อไม่ให้สูญเสียชื่อคอลัมน์ (แก้ปัญหา Warning)
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print(f"ขนาดของชุดข้อมูลฝึกสอน (Training Set): {len(X_train)} คน")
print(f"ขนาดของชุดข้อมูลทดสอบ (Testing Set): {len(X_test)} คน")

**ขั้นตอนที่ 3: ฝึกสอนโมเดลที่ 1 (Logistic Regression)**
แบบจำลอง Logistic Regression ถูกนำมาใช้ในการวิเคราะห์ความน่าจะเป็นที่บุคคลจะตกอยู่ในกลุ่มความเสี่ยงระดับต่างๆ

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# การสร้างแบบจำลอง Logistic Regression
model_lr = LogisticRegression(C=0.001, class_weight='balanced', max_iter=1000, random_state=42)

# การฝึกสอนแบบจำลอง (Model Training)
model_lr.fit(X_train_scaled, y_train)

# การทำนายผลลัพธ์จากชุดข้อมูลทดสอบ (Model Prediction)
y_pred_lr = model_lr.predict(X_test_scaled)

# การประเมินความแม่นยำของแบบจำลอง (Accuracy Score)
acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"ระดับความแม่นยำ (Accuracy) ของ Logistic Regression: {acc_lr*100:.2f}%")

# การแสดงผลการประเมินแบบจำลองโดยละเอียด
print("\nรายงานการประเมินการจำแนกประเภท (Classification Report):")
print(classification_report(y_test, y_pred_lr, target_names=['Safe', 'At Risk']))


**ขั้นตอนที่ 3.1: การสร้างและแสดงผล Confusion Matrix สำหรับ Logistic Regression**
Confusion Matrix คือตารางที่ใช้สรุปผลการทำนายของโมเดล Classification โดยเฉพาะ มันจะช่วยให้เราเห็นภาพว่าโมเดลของเรา **ทายถูก** และ **ทายผิด** อย่างไรบ้างในแต่ละคลาส

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

# สร้าง Confusion Matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
labels = ['Safe', 'At Risk']

# พล็อตกราฟ Heatmap ด้วย Seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='YlGnBu', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix for Logistic Regression')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

**ขั้นตอนที่ 4: ฝึกสอนโมเดลที่ 2 (K-Nearest Neighbors - KNN)**
แบบจำลอง K-Nearest Neighbors (KNN) จะทำการจำแนกกลุ่มความเสี่ยงโดยพิจารณาจากความคล้ายคลึงของพฤติกรรมเทียบกับจุดข้อมูลที่ใกล้ที่สุด

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
# ทดลองค่า k ตั้งแต่ 1 ถึง 20 แล้ววัด Accuracy ของแต่ละค่า
best_k = 1
best_acc = 0
acc_list = []
for k in range(1, 21):
    temp_model = KNeighborsClassifier(n_neighbors=k)
    temp_model.fit(X_train_scaled, y_train)
    temp_pred = temp_model.predict(X_test_scaled)
    acc = accuracy_score(y_test, temp_pred)
    acc_list.append(acc)
    
    if acc > best_acc:
        best_acc = acc
        best_k = k
print(f"ค่า k ที่ดีที่สุด: {best_k}")
print(f"Accuracy สูงสุดที่ได้: {best_acc*100:.2f}%")

In [ ]:
# ใช้ชุดข้อมูลที่ผ่านการปรับสเกลแล้ว (X_train_scaled, X_test_scaled) จากขั้นตอนก่อนหน้า
# การสร้างแบบจำลอง K-Nearest Neighbors
model_knn = KNeighborsClassifier(n_neighbors=best_k)

# การฝึกสอนแบบจำลอง (Model Training)
model_knn.fit(X_train_scaled, y_train)

# การทำนายผลลัพธ์จากชุดข้อมูลทดสอบ (Model Prediction)
y_pred_knn = model_knn.predict(X_test_scaled)

# การประเมินความแม่นยำของแบบจำลอง (Accuracy Score)
acc_knn = accuracy_score(y_test, y_pred_knn)
print(f"ระดับความแม่นยำ (Accuracy) ของ K-Nearest Neighbors (KNN): {acc_knn*100:.2f}%")

# การแสดงผลการประเมินแบบจำลองโดยละเอียด
print("\nรายงานการประเมินการจำแนกประเภท (Classification Report):")
print(classification_report(y_test, y_pred_knn, target_names=['Safe', 'At Risk']))


**ขั้นตอนที่ 4.1: การสร้างและแสดงผล Confusion Matrix สำหรับ K-Nearest Neighbors (KNN)**
เรามาดูตาราง Confusion Matrix สำหรับโมเดล KNN กันบ้าง เพื่อเปรียบเทียบว่าโมเดลนี้มีการทำนายที่แตกต่างจาก Logistic Regression อย่างไร และมีข้อผิดพลาดในคลาสใดบ้าง

In [ ]:
# สร้าง Confusion Matrix
cm_knn = confusion_matrix(y_test, y_pred_knn)
labels = ['Safe', 'At Risk']

# พล็อตกราฟ Heatmap ด้วย Seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='YlGnBu', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix for K-Nearest Neighbors (KNN)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

**ขั้นตอนที่ 5: สรุปผลการเปรียบเทียบโมเดลและเหตุผลในการเลือก**

| เกณฑ์ | Logistic Regression | KNN (k ที่ดีที่สุด) |
|:------|:---:|:---:|
| Accuracy | 80% | 80% |
| อธิบายเหตุผลได้ | บอกได้ว่าปัจจัยไหนสำคัญ | ไม่มี |
| ค่าความน่าจะเป็น | บอกเปอร์เซ็นต์ความเสี่ยงได้ | มีแต่ความแม่นยำน้อยกว่า |
| นำไปใช้จริงได้ | ง่าย เพราะเป็นสูตรคำนวณ | ยาก เพราะต้องเก็บข้อมูลทั้งหมดไว้ |

**สรุปว่าเลือก Logistic Regression:**
เราเลือก Logistic Regression เป็นโมเดลหลัก เพราะโมเดลนี้สามารถบอกได้ว่าปัจจัยไหนส่งผลต่อความเสี่ยงมากที่สุด ซึ่งช่วยให้เราตอบอาจารย์ได้ว่าทำไมคนถึงเสี่ยง ไม่ใช่แค่บอกว่าเสี่ยงหรือไม่เสี่ยงเฉยๆ ครับ


---
# Phase 4: การนำไปใช้และสรุปผล (Deployment & Recommendations)
ส่วนนี้แสดงผลการสกัดคุณลักษณะที่สำคัญ (Feature Importance) และจำลองระบบประเมินความเสี่ยงเพื่อนำเสนอคำแนะนำที่เหมาะสม

**1. ปัจจัยที่มีอิทธิพลที่สุด (Feature Importance จาก Logistic Regression)**
เราจะดูว่าตัวแปรไหนที่ทำให้โมเดลมองว่า 'เสี่ยงสูง' หรือ 'เสี่ยงต่ำ' 

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# ดูค่าน้ำหนักของปัจจัยต่างๆ จากโมเดล
# เราเลือกดูว่าปัจจัยไหนที่ทำให้คนอยู่ในกลุ่มเสี่ยง (At Risk)
importance = model_lr.coef_[0]

# การสร้าง DataFrame สำหรับการแสดงผล
feature_imp = pd.DataFrame({'Feature': features, 'Importance': importance})
feature_imp = feature_imp.sort_values(by='Importance', ascending=True)

# พล็อตกราฟ Horizontal Bar Chart
colors = ['mediumseagreen' if x < 0 else 'indianred' for x in feature_imp['Importance']]
feature_imp.plot(kind='barh', x='Feature', y='Importance', color=colors, figsize=(10, 6), legend=False)

plt.title('ปัจจัยที่ส่งผลต่อ "ความเสี่ยงสูง" (Feature Importance)')
plt.xlabel('ค่าน้ำหนัก (Weight: ลบ=ลดความเสี่ยง, บวก=เพิ่มความเสี่ยง)')
plt.ylabel('ปัจจัยต่างๆ')
plt.axvline(x=0, color='black', linewidth=1)
plt.show()

**2. ฟังก์ชันทำนายผลลัพธ์แบบ Hybrid (Live Demo Mock-up)**
ส่วนนี้ตั้งค่าตัวแปรสมมติ (Persona) เอาไว้ หากต้องการทดสอบเคสอื่นๆ สามารถเปลี่ยนตัวเลขในโค้ดแล้วกดรันได้เลย

In [ ]:
# กำหนดข้อมูลสมมติเพื่อทดสอบโมเดล
mock_data = {
    'income_num': [5000],            # รายได้น้อย
    'debt_score': [3],               # มีหนี้เยอะ
    'liquidity_score': [3],          # ไม่มีเงินสำรอง
    'payment_score': [3],            # จ่ายขั้นต่ำ
    'credit_score': [2],             # มีบัตรหลายใบ
    'assets_score': [3],             # ไม่มีทรัพย์สิน
    'emergency_score': [3],          # ต้องกู้ยืม
    'impulse_buying': [5],           # ซื้อของตามใจบ่อย
    'concern_level': [5],            # กังวลมาก
    'financial_literacy': [1],       # ความรู้น้อย
    'travel_cost': [4000],           
    'family_num': [5]                
}

user_df = pd.DataFrame(mock_data)

# ทำนายผล
user_df_scaled = pd.DataFrame(scaler.transform(user_df), columns=user_df.columns)
prediction_code = model_lr.predict(user_df_scaled)[0]
probabilities = model_lr.predict_proba(user_df_scaled)[0]

label_map = {0: 'Safe (ปลอดภัย)', 1: 'At Risk (มีความเสี่ยง)'}
advice_map = {
    0: "คำแนะนำ: สถานะการเงินดี ควรรักษาวินัยการออมและการลงทุนอย่างต่อเนื่อง",
    1: "คำแนะนำ: อันตราย ควรหยุดก่อหนี้เพิ่มและรีบหาทางสำรองเงินฉุกเฉิน"
}

print("--- ผลการประเมินความเสี่ยง ---")
print(f"ระดับความเสี่ยงที่ประเมินได้: {label_map[prediction_code]}")
print(f"ความน่าจะเป็น: {probabilities[prediction_code]*100:.2f}%")
print("----------------------------")
print(advice_map[prediction_code])